In [1]:
# まず既存のクローンがあれば削除
%cd /content
!rm -rf fx-backtest-strategyH

# 再クローン（空っぽでも構造を復元）
!git clone https://github.com/Taku832/fx-backtest-strategyH.git
%cd fx-backtest-strategyH


/content
Cloning into 'fx-backtest-strategyH'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 28 (delta 5), reused 28 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 8.42 MiB | 15.28 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/fx-backtest-strategyH


In [2]:
!ls -a


.  ..  .config	.git  sample_data


In [3]:
!git init


Reinitialized existing Git repository in /content/fx-backtest-strategyH/.git/


In [5]:
!git remote remove origin


In [9]:
from google.colab import files
uploaded = files.upload()  # 複数選択してアップしてOK


Saving strategyH.py to strategyH.py
Saving USDJPY15_20250613.csv to USDJPY15_20250613 (2).csv


In [10]:
!head strategyH.py


# strategyH.py
import backtrader as bt
import math
from datetime import timedelta

# ──────────────────────────────────────────────
# 1. ZigZag（Depth / Deviation / Backstep）を簡易実装
# ──────────────────────────────────────────────
class ZigZag(bt.Indicator):
    lines = ('zigzag',)


In [11]:
# Gitユーザー情報（必要に応じて一度だけ設定）
!git config --global user.name "Taku832"
!git config --global user.email "mi6_james_007_bond@yahoo.co.jp"  # ← ご自身のGitHub登録メールに置き換え

# ファイルをステージング
!git add strategyH.py USDJPY15_20250613.csv

# コミットメッセージを付けて保存
!git commit -m "Add strategyH.py and USDJPY15_20250613.csv for backtest"

# GitHubにプッシュ（mainブランチ想定）
!git push origin main


[main 375a4a9] Add strategyH.py and USDJPY15_20250613.csv for backtest
 2 files changed, 19329 insertions(+)
 create mode 100644 USDJPY15_20250613.csv
 create mode 100644 strategyH.py
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 274.62 KiB | 3.08 MiB/s, done.
Total 4 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/Taku832/fx-backtest-strategyH.git
   429ea1e..375a4a9  main -> main


In [12]:
# ===== 準備セル =====
import backtrader as bt, pandas as pd, pickle, os, datetime as dt

# CSVの読み込み（アップロード済みなので再アップ不要）
data15 = pd.read_csv("USDJPY15_20250613.csv", parse_dates=["datetime"], index_col="datetime")

# Backtrader形式に変換
data_bt = bt.feeds.PandasData(dataname=data15)

# ===== バックテスト実行セル =====
cerebro = bt.Cerebro()
cerebro.adddata(data_bt, timeframe=bt.TimeFrame.Minutes, compression=15)
cerebro.resampledata(data_bt, timeframe=bt.TimeFrame.Minutes, compression=60)
cerebro.addstrategy(__import__("strategyH").StrategyH)
results = cerebro.run()

# ===== 中間保存セル =====
with open("result_checkpoint.pkl", "wb") as f:
    pickle.dump(results[0].trades_log, f)


ModuleNotFoundError: No module named 'backtrader'

In [13]:
!pip install backtrader


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 7.6 MB/s eta 0:00:00


In [14]:
# ===== 準備セル =====
import backtrader as bt, pandas as pd, pickle, os, datetime as dt

# CSVの読み込み（すでにColabにアップ済みの前提）
data15 = pd.read_csv("USDJPY15_20250613.csv", parse_dates=["datetime"], index_col="datetime")

# ✅ Backtraderデータに変換
data_bt = bt.feeds.PandasData(dataname=data15)

# ===== バックテスト実行セル =====
cerebro = bt.Cerebro()
cerebro.adddata(data_bt, timeframe=bt.TimeFrame.Minutes, compression=15)
cerebro.resampledata(data_bt, timeframe=bt.TimeFrame.Minutes, compression=60)
cerebro.addstrategy(__import__("strategyH").StrategyH)
results = cerebro.run()

# ===== 中間保存セル（trades_log をPickle）=====
with open("result_checkpoint.pkl", "wb") as f:
    pickle.dump(results[0].trades_log, f)

print("✅ バックテスト完了、結果を result_checkpoint.pkl に保存しました。")


ValueError: Missing column provided to 'parse_dates': 'datetime'

In [15]:
import pandas as pd

# CSVファイルの先頭5行を表示して、列名を確認
df = pd.read_csv("USDJPY15_20250613.csv")
print("📌 列名一覧:", df.columns.tolist())
df.head()


📌 列名一覧: ['2022.09.08', '05:30', '143.993', '144.123', '143.992', '144.044', '2476']


,2022.09.08,05:30,143.993,144.123,143.992,144.044,2476
0,2022.09.08,05:45,144.044,144.164,143.957,144.149,1839.0
1,2022.09.08,06:00,144.149,144.209,144.065,144.143,1768.0
2,2022.09.08,06:15,144.142,144.162,144.078,144.087,1912.0
3,2022.09.08,06:30,144.087,144.251,144.077,144.179,1957.0
4,2022.09.08,06:45,144.178,144.240,144.161,144.207,1426.0


In [16]:
import pandas as pd

# 列名を明示的に指定（1行目がヘッダーではない）
columns = ["datetime", "open", "high", "low", "close", "volume"]
df = pd.read_csv("USDJPY15_20250613.csv", names=columns, header=None, parse_dates=["datetime"])
df.set_index("datetime", inplace=True)

# 表示確認
df.head()


/tmp/ipython-input-16-2751756211.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv("USDJPY15_20250613.csv", names=columns, header=None, parse_dates=["datetime"])


,open,high,low,close,volume
datetime,,,,,
2025-07-06 05:30:00,143.993,144.123,143.992,144.044,2476.0
2025-07-06 05:45:00,144.044,144.164,143.957,144.149,1839.0
2025-07-06 06:00:00,144.149,144.209,144.065,144.143,1768.0
2025-07-06 06:15:00,144.142,144.162,144.078,144.087,1912.0
2025-07-06 06:30:00,144.087,144.251,144.077,144.179,1957.0


In [17]:
# Backtrader対応のCSVをColab上に保存
df.to_csv("USDJPY15_20250613_bt.csv")


In [18]:
%cd /content/fx-backtest-strategyH


/content/fx-backtest-strategyH


In [19]:
!ls -l USDJPY15_20250613_bt.csv


-rw-r--r-- 1 root root 1117126 Jul  6 14:02 USDJPY15_20250613_bt.csv


In [20]:
!git add USDJPY15_20250613_bt.csv
!git commit -m "Add Backtrader-ready CSV"
!git push origin main


[main 1418303] Add Backtrader-ready CSV
 1 file changed, 19198 insertions(+)
 create mode 100644 USDJPY15_20250613_bt.csv
Enumerating objects: 4, done.
Counting objects: 100% (4/4), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 270.83 KiB | 2.85 MiB/s, done.
Total 3 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/Taku832/fx-backtest-strategyH.git
   375a4a9..1418303  main -> main


In [21]:
from google.colab import files
files.download("USDJPY15_20250613_bt.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
!rm USDJPY15_20250613.csv


In [23]:
import backtrader as bt
import pandas as pd
import pickle

# 🔽 加工済みCSVの読み込み
df = pd.read_csv("USDJPY15_20250613_bt.csv", parse_dates=["datetime"], index_col="datetime")

# ✅ Backtrader対応データに変換
data15 = bt.feeds.PandasData(dataname=df)

# ✅ Cerebro初期化とデータ登録
cerebro = bt.Cerebro()
cerebro.adddata(data15, timeframe=bt.TimeFrame.Minutes, compression=15)
cerebro.resampledata(data15, timeframe=bt.TimeFrame.Minutes, compression=60)

# ✅ strategyH.py のクラスを読み込んで追加
from strategyH import StrategyH
cerebro.addstrategy(StrategyH)

# ✅ 実行
results = cerebro.run()

# ✅ 中間保存（トレードログをPickleで保存）
with open("result_checkpoint.pkl", "wb") as f:
    pickle.dump(results[0].trades_log, f)


TypeError: Cerebro.adddata() got an unexpected keyword argument 'timeframe'

In [24]:
import backtrader as bt
import pandas as pd
import pickle

# 🔽 加工済みCSVの読み込み
df = pd.read_csv("USDJPY15_20250613_bt.csv", parse_dates=["datetime"], index_col="datetime")

# ✅ Backtraderデータに変換
data15 = bt.feeds.PandasData(dataname=df)

# ✅ Cerebro初期化とデータ登録（timeframeなしで登録）
cerebro = bt.Cerebro()
cerebro.adddata(data15)  # 15分足として登録
data60 = cerebro.resampledata(data15, timeframe=bt.TimeFrame.Minutes, compression=60)  # 60分足にリサンプリング

# ✅ strategyH.py のクラスを読み込んで追加
from strategyH import StrategyH
cerebro.addstrategy(StrategyH)

# ✅ 実行
results = cerebro.run()

# ✅ トレードログの中間保存（Pickle）
with open("result_checkpoint.pkl", "wb") as f:
    pickle.dump(results[0].trades_log, f)


In [25]:
import pandas as pd
import pickle

# ✅ Pickleファイルを読み込む
with open("result_checkpoint.pkl", "rb") as f:
    trades = pickle.load(f)

# ✅ DataFrameに変換
df = pd.DataFrame(trades)

# ✅ datetime列をインデックスに設定（集計に便利）
df["entry_dt"] = pd.to_datetime(df["entry_dt"])
df.set_index("entry_dt", inplace=True)

# ✅ 勝敗列（pips > 0 で勝ちとする）
df["win"] = df["pips"] > 0

# ✅ 集計関数（回数・勝率・pips合計・期待値）
def aggregate(group):
    return pd.Series({
        "取引回数": len(group),
        "勝率": group["win"].mean().round(2),
        "損益(pips)": group["pips"].sum().round(1),
        "期待値": (group["pips"].mean()).round(2)
    })

# ✅ 月別集計
monthly_stats = df.groupby(df.index.to_period("M")).apply(aggregate)

# ✅ 週別集計
weekly_stats = df.groupby(df.index.to_period("W")).apply(aggregate)

# ✅ 日別集計（任意）
daily_stats = df.groupby(df.index.date).apply(aggregate)

# ✅ 表示（必要に応じて変更）
from ace_tools import display_dataframe_to_user
display_dataframe_to_user("月別パフォーマンス", monthly_stats)


KeyError: 'entry_dt'

In [26]:
# Pickleファイル読み込み
import pickle

with open("result_checkpoint.pkl", "rb") as f:
    trades = pickle.load(f)

# リストの要素数確認（＝取引件数）
print(f"取引件数: {len(trades)} 件")

# 内容を数件表示
for i, trade in enumerate(trades[:3]):
    print(f"--- Trade {i+1} ---")
    print(trade)


取引件数: 0 件


In [27]:
import pandas as pd

# 元のCSVファイルを読み込み（ヘッダーなし）
df = pd.read_csv("USDJPY15_20250613.csv", header=None)
df.columns = ["time", "open", "high", "low", "close", "volume"]

# ✅ 期間：2022/09/08 〜 2023/06/13（15分足データで補完）
start = pd.to_datetime("2022-09-08 00:00")
periods = len(df)  # データの行数＝時間足の本数
date_range = pd.date_range(start=start, periods=periods, freq="15min")

# 日時を結合（新しいdatetime列を作成）
df["datetime"] = date_range

# 列の順序を変更（Backtrader用）
df = df[["datetime", "open", "high", "low", "close", "volume"]]

# ✅ datetime列をインデックスに（任意）
df.set_index("datetime", inplace=True)

# 📁 保存（Backtrader形式）
df.to_csv("USDJPY15_20250613_bt.csv")

# ✅ 完了確認
print("✅ 日付補完済CSVを保存しました。行数：", len(df))
df.head()


FileNotFoundError: [Errno 2] No such file or directory: 'USDJPY15_20250613.csv'

In [28]:
from google.colab import files
uploaded = files.upload()  # → ファイル選択画面が出るので、USDJPY15_20250613.csv を選択


Saving USDJPY15_20250613.csv to USDJPY15_20250613.csv


In [29]:
import pandas as pd

# 元のCSVファイルを読み込み（ヘッダーなし）
df = pd.read_csv("USDJPY15_20250613.csv", header=None)
df.columns = ["time", "open", "high", "low", "close", "volume"]

# ✅ 期間：2022/09/08 〜 2023/06/13（15分足データで補完）
start = pd.to_datetime("2022-09-08 00:00")
periods = len(df)  # データの行数＝時間足の本数
date_range = pd.date_range(start=start, periods=periods, freq="15min")

# 日時を結合（新しいdatetime列を作成）
df["datetime"] = date_range

# 列の順序を変更（Backtrader用）
df = df[["datetime", "open", "high", "low", "close", "volume"]]

# ✅ datetime列をインデックスに（任意）
df.set_index("datetime", inplace=True)

# 📁 保存（Backtrader形式）
df.to_csv("USDJPY15_20250613_bt.csv")

# ✅ 完了確認
print("✅ 日付補完済CSVを保存しました。行数：", len(df))
df.head()


ValueError: Length mismatch: Expected axis has 7 elements, new values have 6 elements

In [30]:
import pandas as pd

# 一旦ヘッダー無しで読み込む
df = pd.read_csv("USDJPY15_20250613.csv", header=None)

# 先頭5行を表示して確認
df.head()


,0,1,2,3,4,5,6
0,2022.09.08,05:30,143.993,144.123,143.992,144.044,2476.0
1,2022.09.08,05:45,144.044,144.164,143.957,144.149,1839.0
2,2022.09.08,06:00,144.149,144.209,144.065,144.143,1768.0
3,2022.09.08,06:15,144.142,144.162,144.078,144.087,1912.0
4,2022.09.08,06:30,144.087,144.251,144.077,144.179,1957.0


In [31]:
import pandas as pd

# 元のCSVを読み込み（ヘッダーなし）
df = pd.read_csv("USDJPY15_20250613 (1).csv", header=None)

# 列名を指定
df.columns = ["date", "time", "open", "high", "low", "close", "volume"]

# ✅ datetime列を作成
df["datetime"] = pd.to_datetime(df["date"] + " " + df["time"])

# ✅ datetime列をインデックスに設定
df.set_index("datetime", inplace=True)

# ✅ 不要列を削除
df = df[["open", "high", "low", "close", "volume"]]

# ✅ Backtrader対応のCSVとして保存
df.to_csv("USDJPY15_20250613_bt.csv")


In [32]:
# Git管理対象に追加
!git add USDJPY15_20250613_bt.csv

# コミット（コメントは任意）
!git commit -m "Add corrected USDJPY15_20250613_bt.csv after timestamp fix"

# GitHubへ反映（push）
!git push origin main


[main ec02977] Add corrected USDJPY15_20250613_bt.csv after timestamp fix
 1 file changed, 19198 insertions(+), 19198 deletions(-)
 rewrite USDJPY15_20250613_bt.csv (85%)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 61.01 KiB | 1.27 MiB/s, done.
Total 3 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/Taku832/fx-backtest-strategyH.git
   1418303..ec02977  main -> main


In [33]:
from google.colab import files
files.download("USDJPY15_20250613_bt.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
from google.colab import files
uploaded = files.upload()


Saving USDJPY15_20250613_bt.csv to USDJPY15_20250613_bt (1).csv


In [35]:
import backtrader as bt
import pandas as pd
import pickle

# 🔽 データ読み込み（最新の加工済みCSV）
df = pd.read_csv("USDJPY15_20250613_bt.csv", parse_dates=["datetime"], index_col="datetime")

# ✅ Backtraderデータに変換
data_bt = bt.feeds.PandasData(dataname=df)

# ✅ Cerebro初期化とデータ登録
cerebro = bt.Cerebro()
cerebro.adddata(data_bt)  # ← timeframe/compression の指定は不要
cerebro.resampledata(data_bt, timeframe=bt.TimeFrame.Minutes, compression=60)

# ✅ 戦略を追加
from strategyH import StrategyH
cerebro.addstrategy(StrategyH)

# ✅ 実行
results = cerebro.run()

# ✅ ログ保存（trades_log を Pickleで保存）
with open("result_checkpoint.pkl", "wb") as f:
    pickle.dump(results[0].trades_log, f)


TypeError: StrategyH.is_uptrend_hourly() takes 1 positional argument but 2 were given

In [36]:
# 1. 念のためファイルが存在するか確認（表示されればOK）
!ls strategyH.py


strategyH.py


In [37]:
# 2. Gitに追加
!git add strategyH.py


In [38]:
# 3. コミット（コメントはわかりやすく記述）
!git commit -m "Update: 修正版 strategyH.py をアップロード"


On branch main
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	USDJPY15_20250613 (1).csv
	USDJPY15_20250613 (2).csv
	USDJPY15_20250613_bt (1).csv
	__pycache__/
	result_checkpoint.pkl

nothing added to commit but untracked files present (use "git add" to track)


In [39]:
# 再度 strategyH.py を明示的に追加（他の不要なCSVを除外するため）
!git add strategyH.py


In [40]:
# 改めてコミットを実行
!git commit -m "Fix: 修正済 strategyH.py を追加"


On branch main
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	USDJPY15_20250613 (1).csv
	USDJPY15_20250613 (2).csv
	USDJPY15_20250613_bt (1).csv
	__pycache__/
	result_checkpoint.pkl

nothing added to commit but untracked files present (use "git add" to track)


In [41]:
On branch main
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	USDJPY15_20250613 (1).csv
	USDJPY15_20250613 (2).csv
	USDJPY15_20250613_bt (1).csv
	__pycache__/
	result_checkpoint.pkl

nothing added to commit but untracked files present (use "git add" to track)

SyntaxError: invalid syntax (ipython-input-41-2968221268.py, line 1)

In [42]:
# ① strategyH.py が存在するか確認（ファイル名の打ち間違い防止）
!ls strategyH.py


strategyH.py


In [43]:
# ② Git に追加
!git add strategyH.py


In [44]:
# ③ コミット（メッセージは自由に変更OK）
!git commit -m "Add fixed version of strategyH.py"


On branch main
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	USDJPY15_20250613 (1).csv
	USDJPY15_20250613 (2).csv
	USDJPY15_20250613_bt (1).csv
	__pycache__/
	result_checkpoint.pkl

nothing added to commit but untracked files present (use "git add" to track)


In [45]:
from google.colab import files
uploaded = files.upload()  # 最新 strategyH.py を選択してアップロード


Saving strategyH.py to strategyH (1).py


In [46]:
!rm strategyH.py


In [47]:
!mv "strategyH (1).py" strategyH.py


In [48]:
!git add strategyH.py
!git commit -m "Replace with latest strategyH.py"
!git push origin main


[main c58d586] Replace with latest strategyH.py
 1 file changed, 35 insertions(+), 50 deletions(-)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 793 bytes | 793.00 KiB/s, done.
Total 3 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/Taku832/fx-backtest-strategyH.git
   ec02977..c58d586  main -> main


In [49]:
from strategyH import StrategyH
cerebro.addstrategy(StrategyH)
results = cerebro.run()


TypeError: StrategyH.is_uptrend_hourly() takes 1 positional argument but 2 were given

In [50]:
from google.colab import files
files.upload()  # strategyH.py を選択


Saving strategyH.py to strategyH (1).py


{'strategyH (1).py': b'# strategyH.py  \xe2\x94\x80\xe2\x94\x80 2025-07-06 \xe4\xbf\xae\xe6\xad\xa3\xe7\x89\x88\r\nimport backtrader as bt\r\n\r\n# \xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\r\n# 1. ZigZag\xef\xbc\x88Depth / Deviation / Backstep\xef\xbc\x89\r\n# \xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\r\nclass Z

In [51]:
import shutil

# strategyH (1).py → strategyH.py に上書きコピー
shutil.move("strategyH (1).py", "strategyH.py")


'strategyH.py'

In [52]:
!git add strategyH.py


In [53]:
!git commit -m "Update strategyH.py with latest version (2025-07-07)"


[main 5b08614] Update strategyH.py with latest version (2025-07-07)
 1 file changed, 51 insertions(+), 47 deletions(-)


In [54]:
!git push origin main


Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 1.09 KiB | 1.09 MiB/s, done.
Total 3 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/Taku832/fx-backtest-strategyH.git
   c58d586..5b08614  main -> main


In [55]:
# ✅ Colabが起動直後なら再度読み込みが必要
from strategyH import StrategyH

# ✅ Cerebroに戦略を追加
cerebro.addstrategy(StrategyH)

# ✅ 実行（再スタート）
results = cerebro.run()

# ✅ ログ出力（必要であれば）
for trade in results[0].trades_log:
    print(trade)


TypeError: StrategyH.is_uptrend_hourly() takes 1 positional argument but 2 were given

In [56]:
!ls fx-backtest-strategyH/


ls: cannot access 'fx-backtest-strategyH/': No such file or directory
